# Revue visuelle des jardins — GardnAIze PR #30
Validation individuelle avant constitution des jeux TRAIN/VAL/TEST. Les fichiers images restent locaux; aucun téléchargement automatique.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd()))
from garden_review import load_manifest, load_reviews, save_reviews, reviewed_records, export_json
from IPython.display import display, Image, Markdown
import ipywidgets as widgets
manifest_path = Path('inputs/garden_manifest.csv')
review_path = Path('outputs/garden_reviews.csv')
rows = load_manifest(manifest_path)
reviews = load_reviews(review_path)
print(f'{len(rows)} images; {len(reviews)} décisions enregistrées')

In [ ]:
index = 0
output = widgets.Output()
status = widgets.Dropdown(options=['PENDING','ACCEPTED','REJECTED','UNCERTAIN'],description='Décision')
reason = widgets.Text(description='Motif')
prev_btn = widgets.Button(description='← Précédente')
save_btn = widgets.Button(description='Enregistrer et suivante',button_style='success')
next_btn = widgets.Button(description='Suivante →')
def render():
    row=rows[index]; key=(row['source_key'],row['external_id'])
    decision=reviews.get(key,{})
    status.value=decision.get('review_status','PENDING'); reason.value=decision.get('review_reason','')
    with output:
        output.clear_output(wait=True)
        display(Markdown(f'**{index+1}/{len(rows)}** — {key[0]} / {key[1]} — jardin: {row["garden_group_key"]}'))
        p=Path(row['image_path']).resolve()
        # Restrict images to the explicitly configured local dataset root.
        root=Path('inputs/images').resolve()
        if not p.is_relative_to(root):
            display(Markdown('Chemin hors du dossier images autorisé'))
        elif p.is_file():
            display(Image(filename=str(p),width=640))
        else:
            display(Markdown('Image locale introuvable'))
def move(delta):
    global index
    index=max(0,min(len(rows)-1,index+delta)); render()
def save(_):
    row=rows[index]; key=(row['source_key'],row['external_id'])
    reviews[key]={'source_key':key[0],'external_id':key[1],'review_status':status.value,'review_reason':reason.value}
    save_reviews(review_path,reviews)
    move(1)
prev_btn.on_click(lambda _:move(-1))
next_btn.on_click(lambda _:move(1))
save_btn.on_click(save)
display(output,widgets.HBox([prev_btn,next_btn]),status,reason,save_btn)
if rows: render()

In [ ]:
# Exporter les décisions et le split stable par jardin après la revue.
export_json('outputs/gardens_reviewed.json',reviewed_records(rows,reviews))
print('Export enregistré')